# Кандидатогенерация для поиска услуг Авито

Описание решения - в `README.md`.

In [1]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path('dataset')


In [2]:
def read_parquet(path: Path) -> pd.DataFrame:
    """Читает parquet и приводит типы к удобным.
    Decimal тип данных переводим в float64 для большей быстроты и меньшей памяти
    NaN знвчение заменяем пустыми строками для дальнейшей склейки текстов
    Идентификаторы item_id, query_id переводим в строки, чтобы не потерять ведущие нули
    """
    table = pq.read_table(path)
    fields = [
        pa.field(f.name, pa.float64()) if pa.types.is_decimal(f.type) else f
        for f in table.schema
    ]
    df = table.cast(pa.schema(fields)).to_pandas()

    # Пустая строка для текстовых полей удобнее NaN при склейке текстов
    text_cols = [c for c in df.columns
                 if c.endswith(('_raw', '_text')) or c == 'search_query']
    df[text_cols] = df[text_cols].fillna('')

    # Идентификаторы делаем строками, чтобы не потерять ведущие нули
    for c in ('item_id', 'query_id'):
        if c in df.columns:
            df[c] = df[c].astype(str)
    return df


In [3]:
DATA_DIR = Path('dataset')
train = read_parquet(DATA_DIR / 'train.parquet')               # пара запрос - выбранное объявление
queries = read_parquet(DATA_DIR / 'benchmark_queries.parquet') # запросы, для которых строим ответ
items = read_parquet(DATA_DIR / 'benchmark_items.parquet')     # корпус, в котором ищем

for name, df in [('train', train), ('queries', queries), ('items', items)]:
    mem = df.memory_usage(deep=True).sum() / 2**30
    print(f'{name:8s} shape={df.shape}, память {mem:.2f} ГБ')

# Сверяем размеров с условием задачи и проверяем уникальность идентификаторов
# Доверяй, но проверяй
assert len(train) == 497673
assert len(queries) == 2452 and queries.query_id.is_unique
assert len(items) == 189212 and items.item_id.is_unique


train    shape=(497673, 19), память 3.42 ГБ
queries  shape=(2452, 6), память 0.00 ГБ
items    shape=(189212, 14), память 1.28 ГБ


In [4]:
train.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 497673 entries, 0 to 497672
Data columns (total 19 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   search_query               497673 non-null  object 
 1   search_location_id         497673 non-null  int64  
 2   search_is_delivery_search  497673 non-null  int32  
 3   search_infm_params_text    497673 non-null  object 
 4   search_category            497673 non-null  int64  
 5   item_title_raw             497673 non-null  object 
 6   item_rating_reviews_count  478462 non-null  float64
 7   item_rating                469441 non-null  float64
 8   item_price                 497673 non-null  float64
 9   item_microcat_id           497673 non-null  int64  
 10  item_longitude             497669 non-null  float64
 11  item_location_id           497673 non-null  int64  
 12  item_latitude              497669 non-null  float64
 13  item_is_phone_hidden       49

In [5]:
queries.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2452 entries, 0 to 2451
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   query_id                   2452 non-null   object
 1   search_query               2452 non-null   object
 2   search_location_id         2452 non-null   int64 
 3   search_is_delivery_search  2452 non-null   int32 
 4   search_infm_params_text    2452 non-null   object
 5   search_category            2452 non-null   int64 
dtypes: int32(1), int64(2), object(3)
memory usage: 679.7 KB


In [6]:
items.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189212 entries, 0 to 189211
Data columns (total 14 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   item_title_raw             189212 non-null  object 
 1   item_rating_reviews_count  177597 non-null  float64
 2   item_rating                171581 non-null  float64
 3   item_price                 189212 non-null  float64
 4   item_microcat_id           189212 non-null  int64  
 5   item_longitude             189211 non-null  float64
 6   item_location_id           189212 non-null  int64  
 7   item_latitude              189211 non-null  float64
 8   item_is_phone_hidden       189212 non-null  bool   
 9   item_is_message_forbidden  189212 non-null  bool   
 10  item_infm_params_text      189212 non-null  object 
 11  item_id                    189212 non-null  object 
 12  item_description_raw       189212 non-null  object 
 13  item_category_id           18

In [7]:
train.head(3)

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,скупка телевизоров,652430,0,,114,Скупка б/у техники,91.0,4.989011,1.0,2303374,39.712818,652430,54.629230,True,False,"Вид услуги Место оказания услуг Первомайский пр-т, 66 Тип стоимости за услугу Работаете с юрлица...",e8b685dffe1a408e,"Скупаю практически любую современную новую и б/у технику(обязательно рабочую), до 80% от рыночно...",114
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.0,2303374,44.051010,640860,56.273390,False,False,"Вид услуги Место оказания услуг Нижний Новгород, Советский район, жилой комплекс Новая Кузнечиха...",92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля в Нижнем Новгороде и области.\n\n🫴 Помогу подобра...,114
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье",114,"Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.0,86469,30.128784,653240,59.783688,False,False,"Вид услуги Красота, здоровье Место оказания услуг Санкт-Петербург, садоводческое некоммерческое ...",624846856ce81d69,"В ритме современной жизни так сложно найти момент, чтобы остановиться: выключить телефон, отлож...",114


In [8]:
queries.head()

,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,Вид услуги Деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,"Вид услуги Красота, здоровье",114
3,660ac9QVtXkRxZC3,подъемник 4 х стоечный,662810,0,,114
4,YgHcM9MVbxKnxD1e,монтаж видеодомофонов,642790,0,,114


In [9]:
items.head()

,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,Ремонт/выкуп компьют. и ноутбуков с выездом на дом,57.0,5.0,500.0,2097573,42.047194,631060,44.228180,False,False,Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальна...,111eb8b979577d79,"Выезд на дом в любое время, вплоть до 23:00\n\n● Диагностика устройств;\n\n● Чистка от пыли и за...",114
1,Обучение ребенка чтению,7.0,5.0,900.0,86453,49.627392,631870,58.627609,False,False,"Вид услуги Обучение, курсы Место оказания услуг ул. Чернышевского, 35 Тип услуги Детское развити...",75fc8e10f5a66fc4,"Дорогие родители и маленькие книголюбы! \n\nЯ, детский психолог/нейропсихолог, рада пригласить в...",114
2,Афрокудри 5+,14.0,5.0,1000.0,86467,40.936892,628500,56.990307,True,False,"Вид услуги Красота, здоровье Место оказания услуг городской округ Иваново, Фрунзенский район Тип...",3f6ae81704565b9c,"ВНИМАНИЕ Укладка !!!! 🥳\n\nАФРОкудри отличный вариант замены хим завивки, если хочется поменять ...",114
3,Монтаж малых архитектурных форм,2.0,5.0,5000.0,2058739,37.576183,637640,55.662735,False,False,"Вид услуги Строительство Место оказания услуг Севастопольский пр-т, 28к4 Тип стоимости за услугу...",0618dec37a59a21a,"Оказываем все виды услуг по сборке и монтажу игровых площадок , а так же все виды работ по устро...",114
4,Репетитор по истории 10 класс ЕГЭ,19.0,5.0,1500.0,86456,44.750474,624850,48.786008,False,False,"Вид услуги Обучение, курсы Место оказания услуг Волгоградская обл., Волжский, пл. имени В. И. Ле...",13da2c81574677ed,🚀 Сдай ЕГЭ по истории на 80+ за 6 месяцев!\n\nМечтаешь поступить в вуз своей мечты? ✨\n\nИстория...,114


## Исследовательский анализ данных


In [10]:
# подсчёт доли пропущенных значений
def missing_report(df):
    obj = df.select_dtypes('object')
    return pd.DataFrame({
        'nan_share': df.isna().mean(),
        'empty_str_share': obj.eq('').mean(),
    }).fillna(0).round(3)

pd.concat({
    'train': missing_report(train),
    'queries': missing_report(queries),
    'items': missing_report(items),
}, axis=1).fillna('-')

train                   queries                  \
                          nan_share empty_str_share nan_share empty_str_share   
item_category_id                0.0             0.0         -               -   
item_description_raw            0.0             0.0         -               -   
item_id                         0.0             0.0         -               -   
item_infm_params_text           0.0             0.0         -               -   
item_is_message_forbidden       0.0             0.0         -               -   
item_is_phone_hidden            0.0             0.0         -               -   
item_latitude                   0.0             0.0         -               -   
item_location_id                0.0             0.0         -               -   
item_longitude                  0.0             0.0         -               -   
item_microcat_id                0.0             0.0         -               -   
item_price                      0.0             0.0         -               -   
item_rating                   0.057             0.0         -               -   
item_rating_reviews_count     0.039             0.0         -               -   
item_title_raw                  0.0             0.0         -               -   
search_category                 0.0             0.0       0.0             0.0   
search_infm_params_text         0.0            0.33       0.0           0.631   
search_is_delivery_search       0.0             0.0       0.0             0.0   
search_location_id              0.0             0.0       0.0             0.0   
search_query                    0.0             0.0       0.0             0.0   
query_id                          -               -       0.0             0.0   

                              items                  
                          nan_share empty_str_share  
item_category_id                0.0             0.0  
item_description_raw            0.0             0.0  
item_id                         0.0             0.0  
item_infm_params_text           0.0             0.0  
item_is_message_forbidden       0.0             0.0  
item_is_phone_hidden            0.0             0.0  
item_latitude                   0.0             0.0  
item_location_id                0.0             0.0  
item_longitude                  0.0             0.0  
item_microcat_id                0.0             0.0  
item_price                      0.0             0.0  
item_rating                   0.093             0.0  
item_rating_reviews_count     0.061             0.0  
item_title_raw                  0.0             0.0  
search_category                   -               -  
search_infm_params_text           -               -  
search_is_delivery_search         -               -  
search_location_id                -               -  
search_query                      -               -  
query_id                          -               -

In [11]:
# сравнение длин поисковых запросов между train и benchmark
pd.DataFrame({
    'train': train.search_query.str.split().str.len().describe(),
    'benchmark': queries.search_query.str.split().str.len().describe(),
}).round(2)

,train,benchmark
count,497673.00,2452.0
mean,2.34,3.2
std,1.07,1.3
min,1.00,1.0
25%,2.00,2.0
50%,2.00,3.0
75%,3.00,4.0
max,17.00,10.0


In [12]:
# частота самых встречаемых запросов
train.search_query.value_counts().head(15)

search_query
маникюр                    6540
массаж                     5702
наращивание ресниц         5395
установка кондиционеров    4573
педикюр                    3436
сантехник                  3405
вывоз мусора               3278
эвакуатор                  3263
электрик                   3158
автоэлектрик               2790
покос травы                2477
вспашка земли              2315
манипулятор                2120
грузоперевозки газель      2034
натяжные потолки           2004
Name: count, dtype: int64

In [13]:
# проверка насколько train пересекается с benchmark по текстам запросов и по объявлениям.
train_texts = set(train.search_query)
train_items = set(train.item_id)
pd.Series({
    'доля текстов запросов бенчмарка, которые есть в train': queries.search_query.isin(train_texts).mean(),
    'доля пар train, объявление которых есть в корпусе бенчмарка': train.item_id.isin(set(items.item_id)).mean(),
    'доля объявлений корпуса, которые встречались в train': items.item_id.isin(train_items).mean(),
}).round(3)

доля текстов запросов бенчмарка, которые есть в train          0.370
доля пар train, объявление которых есть в корпусе бенчмарка    0.066
доля объявлений корпуса, которые встречались в train           0.096
dtype: float64

In [14]:
loc_match = (train.search_location_id == train.item_location_id).mean()
print(f'Доля пар train, где локация объявления совпадает с локацией поиска: {loc_match:.3f}')
# Сколько объявлений корпуса в каждой локации
items_per_loc = items.item_location_id.value_counts()
q_loc_items = queries.search_location_id.map(items_per_loc).fillna(0)
print(f'Доля запросов бенчмарка, в локации которых нет объявлений: {(q_loc_items == 0).mean():.3f}')
# Статистика по числу объявлений в локации запроса
q_loc_items.describe().round(0)

Доля пар train, где локация объявления совпадает с локацией поиска: 0.831
Доля запросов бенчмарка, в локации которых нет объявлений: 0.174


count     2452.0
mean      4703.0
std       6681.0
min          0.0
25%        369.0
50%       1638.0
75%       3659.0
max      19543.0
Name: search_location_id, dtype: float64

In [15]:
print(f'Подкатегорий в корпусе: {items.item_microcat_id.nunique()}, в train: {train.item_microcat_id.nunique()}')

# Для каждого запроса приходится доля пар, пришедшихся на самую частую подкатегорию
top_share = (train.groupby('search_query').item_microcat_id
             .agg(lambda s: s.value_counts(normalize=True).iloc[0]))

# Сколько пар в train у каждого запроса
freq = train.search_query.value_counts()

# Статистика по запросам, которые не менее 5 раз встречались.
top_share[freq[freq >= 5].index].describe().round(3)

Подкатегорий в корпусе: 752, в train: 212


count    11583.000
mean         0.818
std          0.198
min          0.061
25%          0.667
50%          0.875
75%          1.000
max          1.000
Name: item_microcat_id, dtype: float64

У половины частых запросов больше 87.5% выбранных объявлений приходятся на одну подкатегорию. То есть у типичного запроса примерно 9 выборов из 10 приходятся на одну подкатегорию.

### Фильтры поиска

Разбираем `search_infm_params_text` регулярным выражением и проверяем, помогают ли фильтры находить объявления.

- Фильтр «Вид услуги» задан у `64%` запросов train и у `34%` запросов бенчмарка.
- Значение фильтра есть в параметрах выбранного объявления в `94-98%` пар, у корпуса «Вид услуги» заполнен в `93.8%` объявлений.
- «Вид услуги» - грубая категория: внутри одного значения до `101` подкатегории.

Совпадение с фильтром используем как добавку к скору, а не как жёсткий отсев.

In [16]:
# Разбор фильтров поиска в словарь {ключ: значение}
FILTER_KEYS = ['Тип услуги автосервиса', 'Тип услуги', 'Вид услуги']
FILTER_RE = re.compile(
    r'(Тип услуги автосервиса|Тип услуги|Вид услуги) (.+?)'
    r'(?= Тип услуги| Вид услуги| Онлайн-запись| Рейтинг пользователя|$)'
)

def parse_filters(text):
    return {key: value for key, value in FILTER_RE.findall(text)}

pd.Series(train.search_infm_params_text.iloc[:3].map(parse_filters).tolist(),
          index=train.search_infm_params_text.iloc[:3])

search_infm_params_text
                                                                                                                                                 {}
Рейтинг пользователя 4 звезды и выше                                                                                                             {}
Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье    {'Тип услуги': 'СПА-услуги, массаж', 'Вид услуги': 'Красота, здоровье'}
dtype: object

In [17]:
# Доля запросов, где задан каждый фильтр, в train и в бенчмарке
def filter_coverage(df):
    params = df.search_infm_params_text
    parsed = params.map(parse_filters)
    cov = {key: parsed.map(lambda d: key in d).mean() for key in FILTER_KEYS}
    cov['Онлайн-запись'] = params.str.contains('Онлайн-запись', regex=False).mean()
    cov['Рейтинг пользователя'] = params.str.contains('Рейтинг пользователя', regex=False).mean()
    return pd.Series(cov)

pd.DataFrame({'train': filter_coverage(train), 'benchmark': filter_coverage(queries)}).round(3)

,train,benchmark
Тип услуги автосервиса,0.054,0.000
Тип услуги,0.352,0.153
Вид услуги,0.639,0.336
Онлайн-запись,0.030,0.007
Рейтинг пользователя,0.002,0.000


In [18]:
# Доля пар, где значение фильтра поиска есть в параметрах выбранного объявления
train_filters = train.search_infm_params_text.map(parse_filters)
filter_match = {}
for key in FILTER_KEYS:
    mask = train_filters.map(lambda d: key in d)
    expected = key + ' ' + train_filters[mask].map(lambda d: d[key])
    found = [e in p for e, p in zip(expected, train.item_infm_params_text[mask])]
    filter_match[key] = {'пар с фильтром': int(mask.sum()), 'значение есть в параметрах объявления': np.mean(found)}
pd.DataFrame(filter_match).T.round(3)

,пар с фильтром,значение есть в параметрах объявления
Тип услуги автосервиса,26702.0,0.955
Тип услуги,175196.0,0.936
Вид услуги,317887.0,0.984


In [19]:
# Доля объявлений, у которых в параметрах заполнен «Вид услуги»
vid_values = sorted(set(train_filters.map(lambda d: d.get('Вид услуги')).dropna())
                    | set(queries.search_infm_params_text.map(parse_filters).map(lambda d: d.get('Вид услуги')).dropna()))
print(f'Уникальных значений «Вид услуги» в фильтрах: {len(vid_values)}')

def has_vid(params):
    return any(f'Вид услуги {v}' in params for v in vid_values)

pd.Series({
    'объявления train': train.item_infm_params_text.sample(20_000, random_state=SEED).map(has_vid).mean(),
    'объявления корпуса': items.item_infm_params_text.sample(20_000, random_state=SEED).map(has_vid).mean(),
}, name='доля с заполненным «Вид услуги»').round(3)

Уникальных значений «Вид услуги» в фильтрах: 27


объявления train      0.958
объявления корпуса    0.938
Name: доля с заполненным «Вид услуги», dtype: float64

In [20]:
# Для каждого значения «Вид услуги»: число пар, число подкатегорий и доля самой частой подкатегории
vid = train_filters.map(lambda d: d.get('Вид услуги'))
vid_purity = (train[vid.notna()].groupby(vid[vid.notna()].rename('Вид услуги')).item_microcat_id
              .agg(n_pairs='size', n_microcat='nunique',
                   top_share=lambda s: s.value_counts(normalize=True).iloc[0]))
vid_purity.sort_values('n_pairs', ascending=False).head(15).round(3)

,n_pairs,n_microcat,top_share
Вид услуги,,,
"Автосервис, аренда",61919,101,0.341
"Красота, здоровье",61273,47,0.292
Ремонт и отделка,40493,92,0.179
"Сад, благоустройство",16728,49,0.375
Ремонт и обслуживание техники,16600,80,0.218
"Обучение, курсы",14903,58,0.198
Строительство,14590,71,0.158
"Праздники, мероприятия",12795,47,0.399
Грузоперевозки,9347,35,0.543


### Совпадение слов запроса и объявления

Хотя бы одно слово запроса есть в полях выбранного объявления в `92.2%` пар, по первым 5 буквам - в `97.1%`. Поэтому BM25 строим по всем полям с лемматизацией, а для пар без общих слов нужен векторный поиск.

In [21]:
# Доля пар, где хотя бы одно слово запроса есть в поле выбранного объявления
TOKEN_RE = re.compile(r'\w+')

def tokens(text, prefix=None):
    words = TOKEN_RE.findall(text.lower())
    return {w[:prefix] if prefix else w for w in words if len(w) >= 3}

pairs = train.sample(20_000, random_state=SEED)
fields = {
    'заголовок': pairs.item_title_raw,
    'параметры': pairs.item_infm_params_text,
    'описание': pairs.item_description_raw,
}
fields['все поля'] = fields['заголовок'] + ' ' + fields['параметры'] + ' ' + fields['описание']

overlap_rows = {}
for prefix, label in [(None, 'целые слова'), (5, 'первые 5 букв')]:
    q_tok = pairs.search_query.map(lambda t: tokens(t, prefix))
    overlap_rows[label] = {
        name: np.mean([bool(q & tokens(t, prefix)) for q, t in zip(q_tok, col)])
        for name, col in fields.items()
    }
pd.DataFrame(overlap_rows).round(3)

,целые слова,первые 5 букв
заголовок,0.714,0.837
параметры,0.573,0.699
описание,0.772,0.898
все поля,0.922,0.971


### Локации

- Если `search_location_id` встречается у объявлений, считаем его городом, иначе областью. Поиск по городу идёт в `89.2%` пар, объявление из того же города - в `93.2%` из них.
- При несовпадении города медиана расстояния `34` км: пользователь выбирает объявление из соседнего города.
- У `427` запросов бенчмарка локация - область без объявлений. Соответствие «область - города объявлений» строим по train.

In [22]:
# Локация поиска город, если она встречается у объявлений, иначе считаем областью
item_locs = set(train.item_location_id) | set(items.item_location_id)
is_city_search = train.search_location_id.isin(item_locs)
city_pairs = train[is_city_search]
print(f'Доля пар train, где локация поиска встречается у объявлений: {is_city_search.mean():.3f}')
print(f'Совпадение города объявления и поиска в таких парах: '
      f'{(city_pairs.search_location_id == city_pairs.item_location_id).mean():.3f}')

# Для областей: сколько пар, в скольких городах объявления и разброс их широты
region_stats = (train[~is_city_search]
                .groupby('search_location_id')
                .agg(n_pairs=('item_id', 'size'),
                     n_item_locations=('item_location_id', 'nunique'),
                     lat_min=('item_latitude', 'min'), lat_max=('item_latitude', 'max')))
print(f'Локаций поиска, которые не встречаются у объявлений: {len(region_stats)}')
region_stats.sort_values('n_pairs', ascending=False).head(10).round(1)

Доля пар train, где локация поиска встречается у объявлений (город): 0.892
Совпадение города объявления и поиска в таких парах: 0.932
Локаций поиска, которые не встречаются у объявлений: 299


,n_pairs,n_item_locations,lat_min,lat_max
search_location_id,,,,
107620,12277,233,-27.5,63.6
621540,6839,590,0.0,69.4
107621,6310,138,-8.6,64.4
637680,1528,169,0.0,60.0
632660,1422,108,0.0,59.9
629990,1024,28,43.3,60.2
651110,1000,56,42.9,55.8
621550,942,40,0.0,56.3
640310,871,48,52.3,60.0


In [23]:
# Расстояние от города поиска до выбранного объявления, когда города не совпали
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 6371 * 2 * np.arcsin(np.sqrt(a))

# Координаты города - медиана координат его объявлений
loc_coords = (
    pd.concat([train[['item_location_id', 'item_latitude', 'item_longitude']],
               items[['item_location_id', 'item_latitude', 'item_longitude']]])
    .query('item_latitude != 0 or item_longitude != 0')
    .groupby('item_location_id')[['item_latitude', 'item_longitude']].median()
    .set_axis(['lat', 'lon'], axis=1)
)
city_mismatch = city_pairs[city_pairs.search_location_id != city_pairs.item_location_id]
city_mismatch = city_mismatch[(city_mismatch.item_latitude != 0) | (city_mismatch.item_longitude != 0)]
search_xy = loc_coords.reindex(city_mismatch.search_location_id).to_numpy()
dist = pd.Series(haversine_km(search_xy[:, 0], search_xy[:, 1],
                              city_mismatch.item_latitude.to_numpy(),
                              city_mismatch.item_longitude.to_numpy())).dropna()
print(f'Количество пар город поиска != город объявления: {len(dist)}')
dist.describe(percentiles=[.25, .5, .75, .9]).round(1)

Пар «город поиска ≠ город объявления»: 30349


count    30349.0
mean       209.6
std        634.8
min          0.0
25%         17.4
50%         34.0
75%         98.3
90%        482.0
max      14773.1
dtype: float64

In [24]:
# Для запросов бенчмарка с локацией "область": сколько объявлений корпуса доступно через города области из train
corpus_loc_counts = items.item_location_id.value_counts()
region_to_item_locs = train[~is_city_search].groupby('search_location_id').item_location_id.agg(set)

q_is_city = queries.search_location_id.isin(corpus_loc_counts.index)
q_region = queries[~q_is_city]
reachable = q_region.search_location_id.map(
    lambda loc: sum(corpus_loc_counts.get(x, 0) for x in region_to_item_locs.get(loc, set())))
print(f'Запросов бенчмарка с локацией "область": {len(q_region)}, уникальных областей: {q_region.search_location_id.nunique()}')
print(f'Доля таких запросов, чья область встречалась в train: {q_region.search_location_id.isin(region_to_item_locs.index).mean():.3f}')
print(f'Медиана объявлений корпуса в городах области: {reachable.median():.0f}')

Запросов бенчмарка с локацией-областью: 427, уникальных областей: 58
Доля таких запросов, чья область встречалась в train: 0.995
Медиана объявлений корпуса в городах области: 73788


### Дубли

`12.9%` объявлений корпуса продублированы в других городах. Описание короче 50 символов у `2.6%`, нулевых координат практически нет.

In [25]:
# Дубли объявлений, короткие описания и нулевые координаты
pd.Series({
    'объявлений с дублем по заголовку и описанию': items.duplicated(['item_title_raw', 'item_description_raw'], keep=False).mean(),
    'то же внутри одного города': items.duplicated(['item_title_raw', 'item_description_raw', 'item_location_id'], keep=False).mean(),
    'объявлений с описанием короче 50 символов': (items.item_description_raw.str.len() < 50).mean(),
    'объявлений корпуса с нулевыми координатами': ((items.item_latitude == 0) & (items.item_longitude == 0)).mean(),
    'объявлений train с нулевыми координатами': ((train.item_latitude == 0) & (train.item_longitude == 0)).mean(),
}).round(3)

объявлений с дублем по заголовку и описанию    0.129
то же внутри одного города                     0.000
объявлений с описанием короче 50 символов      0.026
объявлений корпуса с нулевыми координатами     0.000
объявлений train с нулевыми координатами       0.000
dtype: float64

## Локальная валидация

- В train нет `query_id`: запрос собираем группировкой по всем колонкам `search_*`.
- Откладываем `3000` запросов: по одному на уникальный текст, из локаций бенчмарка, с весами под распределение локаций бенчмарка.
- Длина запроса, доля известных текстов, доля областей и размер пула в локации близки к бенчмарку.
- Корпус валидации - корпус бенчмарка + релевантные объявления отложенных запросов, которых в нём нет.
- Метрика - Recall50, нижняя граница - 50 случайных объявлений из локации запроса.

In [26]:
# Колонки, которые вместе описывают поисковый запрос: текст, локация, доставка, фильтры, категория
SEARCH_COLS = ['search_query', 'search_location_id', 'search_is_delivery_search',
               'search_infm_params_text', 'search_category']
N_VAL_QUERIES = 3000


# возвращение порядкого номера группы для каждой строки
train['query_key'] = train.groupby(SEARCH_COLS, sort=False).ngroup()
query_table = train.drop_duplicates('query_key')[['query_key'] + SEARCH_COLS]

# Доля каждой локации среди запросов бенчмарка
bench_loc_share = queries.search_location_id.value_counts(normalize=True)
# Оставляем запросы только из локаций бенчмарка, берём по 1 запросу на каждый текст
one_per_text = (query_table[query_table.search_location_id.isin(bench_loc_share.index)]
                .groupby('search_query').sample(1, random_state=SEED))
loc_weight = one_per_text.search_location_id.map(
    bench_loc_share / one_per_text.search_location_id.value_counts(normalize=True))
val_queries = (one_per_text.sample(N_VAL_QUERIES, weights=loc_weight, random_state=SEED)
               .reset_index(drop=True))

# Маски вместо копий train, чтобы не удваивать память
is_val_row = train.query_key.isin(val_queries.query_key)
val_relevant = train.loc[is_val_row].groupby('query_key').item_id.agg(set)
is_fit_row = ~is_val_row

print(f'Запросов в train: {len(query_table)}, в валидации: {len(val_queries)}')
print(f'Строк train для обучения: {is_fit_row.sum()}, релевантных объявлений на запрос валидации: '
      f'{val_relevant.str.len().mean():.2f}')

Запросов в train: 354463, в валидации: 3000
Строк train для обучения: 494252, релевантных объявлений на запрос валидации: 1.10


In [27]:
# Сравнение профиля запросов валидации и бенчмарка
def query_profile(df, fit_texts):
    return pd.Series({
        'слов в запросе': df.search_query.str.split().str.len().mean(),
        'текст есть в обучающей части': df.search_query.isin(fit_texts).mean(),
        'фильтры пустые': (df.search_infm_params_text == '').mean(),
        'локация-область': (~df.search_location_id.isin(item_locs)).mean(),
        'search_category = 0': (df.search_category == 0).mean(),
        'медиана объявлений корпуса в локации': df.search_location_id.map(corpus_loc_counts).fillna(0).median(),
    })

fit_texts = set(train.loc[is_fit_row, 'search_query'])
pd.DataFrame({
    'валидация': query_profile(val_queries, fit_texts),
    'бенчмарк': query_profile(queries, set(train.search_query)),
}).round(3)

,валидация,бенчмарк
слов в запросе,3.082,3.197
текст есть в обучающей части,0.399,0.370
фильтры пустые,0.475,0.631
локация-область,0.183,0.173
search_category = 0,0.000,0.091
медиана объявлений корпуса в локации,1583.000,1638.000


In [28]:
# Корпус валидации: корпус бенчмарка + релевантные объявления отложенных запросов, которых в нём нет
extra_items = (train.loc[is_val_row & ~train.item_id.isin(items.item_id), items.columns]
               .drop_duplicates('item_id'))
val_items = pd.concat([items, extra_items], ignore_index=True)
print(f'Корпус валидации: {len(items)} объявлений бенчмарка + {len(extra_items)} релевантных из train = {len(val_items)}')
assert val_items.item_id.is_unique

Корпус валидации: 189212 объявлений бенчмарка + 3008 релевантных из train = 192220


In [52]:
# Recall@K: доля релевантных объявлений в топ-K, усреднённая по запросам
def recall_at_k(predictions, relevant, k=50):
    scores = [len(set(predictions.get(key, [])[:k]) & rel) / len(rel)
              for key, rel in relevant.items()]
    return float(np.mean(scores))

val_loc_items = val_items.groupby('item_location_id').item_id.agg(list)
rng = np.random.default_rng(SEED)
random_in_city = {
    key: list(rng.permutation(val_loc_items.get(loc, []))[:50])
    for key, loc in zip(val_queries.query_key, val_queries.search_location_id)
}
print(f'Recall50, 50 случайных объявлений из города запроса: {recall_at_k(random_in_city, val_relevant):.4f}')

Recall50, 50 случайных объявлений из города запроса: 0.0430


## BM25

Ссылка на объяснение алгоритма:
https://isakov.work/seo-slovar/bm25/

**Лемматизация**
Лемматизация корпуса занимает около 2 минут, поэтому результат сохраняется в папку `cache`.

In [30]:
import pickle
from functools import lru_cache

import pymorphy3
from sklearn.feature_extraction.text import CountVectorizer

CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)

morph = pymorphy3.MorphAnalyzer()
WORD_RE = re.compile(r'[а-яa-z0-9]+')

@lru_cache(maxsize=None)
def lemmatize(word):
    return morph.parse(word)[0].normal_form.replace('ё', 'е')

def tokenize(text):
    return [lemmatize(w) for w in WORD_RE.findall(text.lower().replace('ё', 'е'))]

tokens_path = CACHE_DIR / f'item_tokens_{len(val_items)}.pkl'
if tokens_path.exists():
    item_tokens = pickle.loads(tokens_path.read_bytes())
else:
    item_tokens = [tokenize(f'{t} {p} {d}') for t, p, d in zip(
        val_items.item_title_raw, val_items.item_infm_params_text, val_items.item_description_raw)]
    tokens_path.write_bytes(pickle.dumps(item_tokens))

print(tokenize('Ремонт принтеров и МФУ с выездом на дом'))
print(f'Слов в корпусе: {sum(map(len, item_tokens))}')

['ремонт', 'принтер', 'и', 'мфу', 'с', 'выезд', 'на', 'дом']
Слов в корпусе: 64391477


**Индекс BM25**

Для каждой пары «слово - объявление» считается вес BM25: чем реже слово в корпусе и чем чаще в объявлении, тем больше вес. Длинные объявления штрафуются, чтобы описание на несколько экранов не выигрывало только за счёт объёма. Скор объявления по запросу - сумма весов слов запроса.

Скор объявления $d$ по запросу $q$:

$$
\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}
$$

$$
\text{IDF}(t) = \ln\left(1 + \frac{N - n(t) + 0.5}{n(t) + 0.5}\right)
$$

где:
- $t$ - уникальное слово запроса после лемматизации,
- $f(t, d)$ - сколько раз слово $t$ встречается в объявлении $d$,
- $|d|$ - число слов в объявлении, $\text{avgdl}$ - среднее число слов по корпусу,
- $N$ - число объявлений в корпусе, $n(t)$ - число объявлений, содержащих слово $t$,
- $k_1 = 1.2$ ограничивает вклад частоты слова, $b = 0.75$ задаёт силу штрафа за длину объявления.

Дробь для каждой пары «слово - объявление» считается заранее и хранится в разреженной матрице, при поиске складываются только столбцы слов запроса.

In [31]:
def bm25_weights(tf, k1=1.2, b=0.75):
    tf = tf.tocsr().astype(np.float32)
    n_docs = tf.shape[0]
    doc_len = np.asarray(tf.sum(axis=1)).ravel()
    df = np.bincount(tf.indices, minlength=tf.shape[1])
    idf = np.log(1 + (n_docs - df + 0.5) / (df + 0.5)).astype(np.float32)
    rows = np.repeat(np.arange(n_docs), np.diff(tf.indptr))
    tf.data = idf[tf.indices] * tf.data * (k1 + 1) / (tf.data + k1 * (1 - b + b * doc_len[rows] / doc_len.mean()))
    return tf.tocsc()

vectorizer = CountVectorizer(analyzer=lambda tokens: tokens)
bm25_matrix = bm25_weights(vectorizer.fit_transform(item_tokens))
print(f'Матрица BM25: {bm25_matrix.shape[0]} объявлений x {bm25_matrix.shape[1]} слов')

def bm25_scores(text):
    term_ids = [vectorizer.vocabulary_[t] for t in set(tokenize(text)) if t in vectorizer.vocabulary_]
    if not term_ids:
        return np.zeros(bm25_matrix.shape[0], dtype=np.float32)
    return np.asarray(bm25_matrix[:, term_ids].sum(axis=1)).ravel()

Матрица BM25: 192220 объявлений x 405119 слов


**Пул кандидатов и скоры**

- Пул запроса: объявления его города и соседних городов, на которые в train приходится не меньше `0.5%` выборов. Для области - такие же частые города области.
- Для каждого кандидата считаем BM25 по тексту запроса и по значениям фильтров.
- Итоговый скор: `text + filter_weight * filters + loc_bonus * same_loc`.

In [45]:
# Пул кандидатов: объявления локации запроса и соседних городов
item_loc_arr = val_items.item_location_id.to_numpy()
rows_by_loc = pd.Series(item_loc_arr).groupby(item_loc_arr).indices

NEIGHBOR_MIN_SHARE = 0.005

fit_cooc = train.loc[is_fit_row].groupby(['search_location_id', 'item_location_id']).size()
# Сосед - город, на который приходится не меньше 0.5% выборов при поиске из данной локации
cooc_share = fit_cooc / fit_cooc.groupby(level=0).transform('sum')
loc_neighbors = {loc: grp.index.get_level_values(1).tolist()
                 for loc, grp in cooc_share[cooc_share >= NEIGHBOR_MIN_SHARE].groupby(level=0)}

def candidate_rows(loc, n_rows, neighbors):
    locs = ([loc] if loc in rows_by_loc else []) + neighbors.get(loc, [])
    rows = [rows_by_loc[l] for l in set(locs) if l in rows_by_loc]
    rows = np.unique(np.concatenate(rows)) if rows else np.arange(n_rows)
    return rows[rows < n_rows]

pool_sizes = pd.Series([len(candidate_rows(loc, len(val_items), loc_neighbors)) for loc in val_queries.search_location_id])
pool_sizes.describe(percentiles=[.25, .5, .75]).round(0)

count     3000.0
mean     13038.0
std      17712.0
min          3.0
25%       2244.0
50%       4576.0
75%      20724.0
max      87420.0
dtype: float64

In [ ]:
# Скоры BM25 по тексту запроса и по фильтрам для всего пула
# собирает значение фильтров в строку
def filter_text(params):
    return ' '.join(value for _, value in FILTER_RE.findall(params))

def bm25_candidates(keys, qdf, n_rows, neighbors):
    result = {}
    for key, text, params, loc in zip(keys, qdf.search_query, qdf.search_infm_params_text, qdf.search_location_id):
        rows = candidate_rows(loc, n_rows, neighbors)
        result[key] = {
            'rows': rows, # номер объявления пула
            'text': bm25_scores(text)[rows], # BM25 по тексту запроса
            'filters': bm25_scores(filter_text(params))[rows], # BM25 по значению фильтров
            'same_loc': item_loc_arr[rows] == loc, # same_loc - если объявление из локации запроса
        }
    return result

t0 = time.time()
val_cands = bm25_candidates(val_queries.query_key, val_queries, len(val_items), loc_neighbors)
print(f'Скоры для {len(val_cands)} запросов: {time.time() - t0:.0f} с')

Скоры для 3000 запросов: 37 с


In [ ]:
# Считаем итоговый скор: text + filter_weight * filters + loc_bonus * same_loc
def bm25_top(cands, filter_weight=0.0, loc_bonus=0.0, own_loc_only=False, k=50):
    preds = {}
    for key, c in cands.items():
        score = c['text'] + filter_weight * c['filters'] + loc_bonus * c['same_loc']
        if own_loc_only and c['same_loc'].any():
            score = np.where(c['same_loc'], score, -np.inf)
        # берём 50 наилучших
        top = np.argsort(-score)[:k]
        preds[key] = val_items.item_id.to_numpy()[c['rows'][top]].tolist()
    return preds

variants = {
    'только своя локация': dict(own_loc_only=True),
    'локация + соседи, без бонуса': dict(),
    'локация + соседи, бонус 20': dict(loc_bonus=20),
    'бонус 20 + фильтры 0.3': dict(loc_bonus=20, filter_weight=0.3),
    'бонус 20 + фильтры 0.5': dict(loc_bonus=20, filter_weight=0.5),
    'бонус 20 + фильтры 1.0': dict(loc_bonus=20, filter_weight=1.0),
}
pd.Series({name: recall_at_k(bm25_top(val_cands, **kw), val_relevant) for name, kw in variants.items()},
          name='Recall50').round(4)

только своя локация             0.8275
локация + соседи, без бонуса    0.8143
локация + соседи, бонус 20      0.8298
бонус 20 + фильтры 0.3          0.8473
бонус 20 + фильтры 0.5          0.8498
бонус 20 + фильтры 1.0          0.8516
Name: Recall50, dtype: float64

In [ ]:
BM25_PARAMS = dict(filter_weight=1.0, loc_bonus=20)
bm25_val_preds = bm25_top(val_cands, **BM25_PARAMS)
print(f'BM25, итоговая конфигурация, Recall50: {recall_at_k(bm25_val_preds, val_relevant):.4f}')

BM25, итоговая конфигурация, Recall50: 0.8516


**Выводы по BM25**

- Лучшая конфигурация: фильтры с весом `1.0` и бонус `20` за свою локацию, `Recall50=0.8516`.
- Значения фильтров дают больше всего: `+0.0218`.
- Соседние города полезны только с бонусом своей локации, без него Recall50 падает до `0.8143`.
- `k1` и `b` почти не влияют, Recall50 в пределах `0.822-0.828`.

## Ответ на бенчмарк

Строим ответ на запросы бенчмарка той же схемой BM25, что и на валидации. Отличий два.

- Ищем только в корпусе бенчмарка `n_rows=len(items)`, отсекаем объявления, добавленные для валидации. Они стоят в конце `val_items`, поэтому первые `len(items)` строк совпадают с корпусом.
- Соседние города считаем по всему train. На валидации мы прятали 3000 отложенных запросов, чтобы статистика соседства не знала их ответов. Для бенчмарка прятать нечего, ответов в train нет, поэтому берём все строки, чтобы статистика была больше.


In [57]:
# Ответ BM25 на запросы бенчмарка
full_cooc = train.groupby(['search_location_id', 'item_location_id']).size()
full_share = full_cooc / full_cooc.groupby(level=0).transform('sum')
full_loc_neighbors = {loc: grp.index.get_level_values(1).tolist()
                      for loc, grp in full_share[full_share >= NEIGHBOR_MIN_SHARE].groupby(level=0)}

bench_cands = bm25_candidates(queries.query_id, queries, len(items), full_loc_neighbors)
bench_preds_bm25 = bm25_top(bench_cands, **BM25_PARAMS)

pool = pd.Series({key: len(c['rows']) for key, c in bench_cands.items()})
print(f'Запросов: {len(bench_preds_bm25)}, медиана пула кандидатов: {pool.median():.0f}, '
      f'запросов с пулом меньше 50: {(pool < 50).sum()}')

Запросов: 2452, медиана пула кандидатов: 3732, запросов с пулом меньше 50: 0


In [ ]:
# Сохранение answer.csv и проверка формата
def save_answer(preds, path):
    pd.DataFrame({
        'query_id': list(preds.keys()),
        'answer': [' '.join(ids) for ids in preds.values()],
    }).to_csv(path, index=False)

def check_answer(path):
    ans = pd.read_csv(path, dtype=str, keep_default_na=False)
    ids = ans.answer.str.split(' ')
    corpus_ids = set(items.item_id)
    assert list(ans.columns) == ['query_id', 'answer']
    assert ans.query_id.is_unique and set(ans.query_id) == set(queries.query_id)
    assert ans.query_id.str.len().eq(16).all()
    assert ids.map(len).le(50).all() and ids.map(lambda x: len(x) == len(set(x))).all()
    assert ids.map(lambda x: all(i in corpus_ids for i in x)).all()
    assert ids.explode().str.fullmatch(r'[0-9a-f]{16}').all()
    print(f'{path}: формат верный, строк {len(ans)}, объявлений в строке в среднем {ids.map(len).mean():.1f}')

save_answer(bench_preds_bm25, 'answer.csv')
check_answer('answer.csv')

## Векторный поиск

BM25 не находит объявления без общих слов с запросом. Для них используем векторный поиск на модели `cointegrated/rubert-tiny2`: она в 5-6 раз быстрее `multilingual-e5-small` на CPU, поэтому её можно дообучить на парах из train.

- Дообучаем на парах «запрос - выбранное объявление» из обучающей части train, негативы - остальные объявления в батче.
- Текст запроса: запрос + значения фильтров. Текст объявления: заголовок + значения фильтров + начало описания.
- Обучение занимает около 100 минут, кодирование корпуса - около 18 минут, результаты сохраняются в `models/` и `cache/`.

In [69]:
import torch
import torch.nn.functional as F
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer, models as st_models
from transformers import AutoModel, AutoTokenizer

BASE_MODEL_DIR = Path('models/rubert-tiny2')
TUNED_MODEL_DIR = Path('models/rubert-tiny2-avito')
if not BASE_MODEL_DIR.exists():
    snapshot_download('cointegrated/rubert-tiny2', local_dir=BASE_MODEL_DIR,
                      allow_patterns=['*.json', 'model.safetensors', '*.txt', '1_Pooling/*'])

def dense_query_text(qdf):
    return (qdf.search_query + '. ' + qdf.search_infm_params_text.map(filter_text)).tolist()

def dense_item_text(idf):
    return (idf.item_title_raw + '. ' + idf.item_infm_params_text.map(filter_text) + '. '
            + idf.item_description_raw.str[:300]).tolist()

def load_encoder(path, max_len=128):
    transformer = st_models.Transformer(str(path), max_seq_length=max_len)
    pooling = st_models.Pooling(transformer.get_embedding_dimension(), 'cls')
    return SentenceTransformer(modules=[transformer, pooling, st_models.Normalize()], device='cpu')

**Дообучение**

Берём до 3 пар на текст запроса, всего около 120 тыс. пар. Одна эпоха, батч 64, `lr=5e-5` с разогревом на 10% шагов.

In [66]:
pair_rows = (train.loc[is_fit_row, ['search_query', 'search_infm_params_text', 'item_id', 'item_title_raw',
                                    'item_infm_params_text', 'item_description_raw']]
             .drop_duplicates(['search_query', 'item_id'])
             .sample(frac=1, random_state=SEED)
             .groupby('search_query').head(3).head(150_000))
dense_pairs = pd.DataFrame({'anchor': dense_query_text(pair_rows), 'positive': dense_item_text(pair_rows)})
del pair_rows
print(f'Пар для обучения: {len(dense_pairs)}')

def finetune_encoder(pairs, out_dir, batch=64, lr=5e-5, scale=20.0):
    torch.manual_seed(SEED)
    tok = AutoTokenizer.from_pretrained(BASE_MODEL_DIR)
    model = AutoModel.from_pretrained(BASE_MODEL_DIR)
    model.train()

    def embed(texts, max_len):
        batch_tok = tok(texts, padding=True, truncation=True, max_length=max_len, return_tensors='pt')
        return F.normalize(model(**batch_tok).last_hidden_state[:, 0], dim=-1)

    steps = len(pairs) // batch
    warmup = int(0.1 * steps)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / warmup if s < warmup else max(0.0, (steps - s) / (steps - warmup)))
    order = np.random.default_rng(SEED).permutation(len(pairs))
    for step in range(steps):
        idx = order[step * batch:(step + 1) * batch]
        q = embed(pairs.anchor.values[idx].tolist(), 32)
        d = embed(pairs.positive.values[idx].tolist(), 128)
        # i-й запрос должен быть ближе всего к i-му объявлению среди всех объявлений батча
        loss = F.cross_entropy(scale * q @ d.T, torch.arange(len(idx)))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad()
        if (step + 1) % 500 == 0:
            print(f'шаг {step + 1}/{steps}, loss {loss.item():.3f}')
    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)

TUNED_MODEL_URL = ('https://github.com/denic-t/Avito_Bootcamp_2_stage_NLP/releases/download/'
                   'model-v1/rubert-tiny2-avito.zip')
TUNED_MODEL_SHA256 = '8c68f8af45ef2b2dcdec3bca50693b6c4449f9948611732d3bce65dbc0a89821'

def download_tuned_model():
    import hashlib, io, urllib.request, zipfile
    try:
        with urllib.request.urlopen(TUNED_MODEL_URL, timeout=120) as resp:
            zipfile.ZipFile(io.BytesIO(resp.read())).extractall(TUNED_MODEL_DIR.parent)
    except Exception as e:
        print(f'Веса не скачались ({e}), модель будет обучена заново')
        return
    weights_hash = hashlib.sha256((TUNED_MODEL_DIR / 'model.safetensors').read_bytes()).hexdigest()
    assert weights_hash == TUNED_MODEL_SHA256, 'скачанные веса отличаются от обученных'

# Готовые веса из GitHub Release экономят ~100 минут обучения, без них модель обучается тем же кодом
if not TUNED_MODEL_DIR.exists():
    download_tuned_model()
if not TUNED_MODEL_DIR.exists():
    finetune_encoder(dense_pairs, TUNED_MODEL_DIR)
print(f'Модель: {TUNED_MODEL_DIR}')

Пар для обучения: 120172
Модель: models\rubert-tiny2-avito


**Кодирование и оценка**

Кодируем корпус и запросы, для каждого кандидата из пула считаем косинусную близость. Сравниваем векторный поиск отдельно и простое объединение с BM25: первые `50 - n_dense` мест из BM25, остальные - из векторного поиска.

In [70]:
encoder = load_encoder(TUNED_MODEL_DIR)
emb_path = CACHE_DIR / f'item_emb_tiny2_avito_{len(val_items)}.npy'
if emb_path.exists():
    item_emb = np.load(emb_path)
else:
    item_emb = encoder.encode(dense_item_text(val_items), batch_size=128, convert_to_numpy=True).astype(np.float32)
    np.save(emb_path, item_emb)

def add_dense_scores(cands, qdf, keys):
    q_emb = encoder.encode(dense_query_text(qdf), batch_size=128, convert_to_numpy=True).astype(np.float32)
    for key, qe in zip(keys, q_emb):
        cands[key]['dense'] = item_emb[cands[key]['rows']] @ qe

add_dense_scores(val_cands, val_queries, val_queries.query_key)
item_id_arr = val_items.item_id.to_numpy()

def bm25_order(c):
    return np.argsort(-(c['text'] + BM25_PARAMS['filter_weight'] * c['filters'] + BM25_PARAMS['loc_bonus'] * c['same_loc']))

def dense_order(c, loc_bonus=0.3):
    return np.argsort(-(c['dense'] + loc_bonus * c['same_loc']))

def union_top(cands, n_dense, k=50):
    preds = {}
    for key, c in cands.items():
        b, d = c['rows'][bm25_order(c)], c['rows'][dense_order(c)]
        merged = list(dict.fromkeys([*b[:k - n_dense], *d[:n_dense], *b[k - n_dense:k]]))[:k]
        preds[key] = item_id_arr[merged].tolist()
    return preds

dense_only = {key: item_id_arr[c['rows'][dense_order(c)[:50]]].tolist() for key, c in val_cands.items()}
pd.Series({
    'BM25': recall_at_k(bm25_top(val_cands, **BM25_PARAMS), val_relevant),
    'векторный поиск': recall_at_k(dense_only, val_relevant),
    'BM25 40 + векторный 10': recall_at_k(union_top(val_cands, 10), val_relevant),
    'BM25 30 + векторный 20': recall_at_k(union_top(val_cands, 20), val_relevant),
    'BM25 20 + векторный 30': recall_at_k(union_top(val_cands, 30), val_relevant),
}, name='Recall50').round(4)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BM25                      0.8516
векторный поиск           0.8084
BM25 40 + векторный 10    0.8786
BM25 30 + векторный 20    0.8833
BM25 20 + векторный 30    0.8821
Name: Recall50, dtype: float64

**Выводы по векторному поиску**

- Без дообучения `rubert-tiny2` даёт только `0.3279`, после дообучения - `0.8084`.
- Векторный поиск находит другие объявления, чем BM25: объединение 30 + 20 даёт `0.8833`, это `+0.0317` к BM25.
- В топ-300 BM25 и векторного поиска вместе попадает `93.2%` релевантных объявлений, поэтому следующий шаг - ранкер на объединённом пуле.

## Ранкер

Пул запроса - объединение топ-200 BM25 и топ-200 векторного поиска. LightGBM с целевой функцией `lambdarank` переупорядочивает пул и отдаёт топ-50.

- Признаки: скоры и места в BM25 и векторном поиске, своя локация и доля выборов в город объявления, совпадение с фильтрами, вероятность подкатегории объявления по тексту запроса, его леммам и фильтру «Вид услуги», рейтинг, отзывы, цена и длина текстов объявления.
- Вероятности подкатегорий и соседство городов для валидации считаем по обучающей части train, для бенчмарка - по всему train.
- Ранкер обучаем на запросах валидации: качество оцениваем на двух фолдах, итоговую модель обучаем на всех 3000 запросах.

In [71]:
import lightgbm as lgb

N_BM25_POOL, N_DENSE_POOL = 200, 200

def microcat_priors(rows_mask):
    df = train.loc[rows_mask, ['search_query', 'search_infm_params_text', 'item_microcat_id']]
    text_mc = df.groupby(['search_query', 'item_microcat_id']).size()
    lemma_mc = (text_mc.rename('n').reset_index()
                .assign(lemma=lambda d: d.search_query.map(lambda t: list(set(tokenize(t)))))
                .explode('lemma').groupby(['lemma', 'item_microcat_id']).n.sum())
    vid = df.search_infm_params_text.map(lambda p: parse_filters(p).get('Вид услуги'))
    vid_mc = df.assign(vid=vid).dropna(subset=['vid']).groupby(['vid', 'item_microcat_id']).size()
    return {
        'text': (text_mc / text_mc.groupby(level=0).transform('sum')).to_dict(),
        'text_cnt': text_mc.groupby(level=0).sum().to_dict(),
        'lemma': (lemma_mc / lemma_mc.groupby(level=0).transform('sum')).to_dict(),
        'lemmas': set(lemma_mc.index.get_level_values(0)),
        'vid': (vid_mc / vid_mc.groupby(level=0).transform('sum')).to_dict(),
    }

fit_priors = microcat_priors(is_fit_row)
full_priors = microcat_priors(np.ones(len(train), dtype=bool))

it_mc = val_items.item_microcat_id.to_numpy()
it_params = val_items.item_infm_params_text.to_numpy()
title_lemmas = [set(tokenize(t)) for t in val_items.item_title_raw]
ITEM_NUM_COLS = ['rating', 'reviews', 'price', 'phone_hidden', 'msg_forbidden', 'title_len', 'desc_len']
item_num = pd.DataFrame({
    'rating': val_items.item_rating,
    'reviews': np.log1p(val_items.item_rating_reviews_count.fillna(0)),
    'price': np.log1p(val_items.item_price.clip(lower=0).fillna(0)),
    'phone_hidden': val_items.item_is_phone_hidden.astype(np.int8),
    'msg_forbidden': val_items.item_is_message_forbidden.astype(np.int8),
    'title_len': val_items.item_title_raw.str.len(),
    'desc_len': np.log1p(val_items.item_description_raw.str.len()),
})[ITEM_NUM_COLS].to_numpy(dtype=np.float64)
print(f'Пар «текст - подкатегория»: {len(fit_priors["text"])}, «лемма - подкатегория»: {len(fit_priors["lemma"])}')

Пар «текст - подкатегория»: 98439, «лемма - подкатегория»: 60157


In [72]:
def ranker_features(cands, qdf, keys, priors, loc_share, relevant=None):
    frames = []
    for key, text, params, loc in zip(keys, qdf.search_query, qdf.search_infm_params_text, qdf.search_location_id):
        c = cands[key]
        ob, od = bm25_order(c), dense_order(c)
        pos = np.unique(np.concatenate([ob[:N_BM25_POOL], od[:N_DENSE_POOL]]))
        rank_b, rank_d = np.empty(len(ob), np.int32), np.empty(len(od), np.int32)
        rank_b[ob], rank_d[od] = np.arange(len(ob)), np.arange(len(od))
        rows = c['rows'][pos]
        mcs = it_mc[rows]
        q_lem = set(tokenize(text))
        filt = parse_filters(params)
        lemma_p = [np.array([priors['lemma'].get((l, m), 0.0) for m in mcs]) for l in q_lem if l in priors['lemmas']]
        f = pd.DataFrame({
            'key': key, 'row': rows,
            'bm25_text': c['text'][pos], 'bm25_filters': c['filters'][pos],
            'bm25_rank': rank_b[pos], 'dense_rank': rank_d[pos],
            'bm25_text_rel': c['text'][pos] / (c['text'].max() + 1e-6),
            'dense': c['dense'][pos], 'dense_rel': c['dense'][pos] - c['dense'].max(),
            'same_loc': c['same_loc'][pos].astype(np.int8),
            'loc_share': [loc_share.get((loc, l), 0.0) for l in item_loc_arr[rows]],
            'mc_text_p': [priors['text'].get((text, m), 0.0) for m in mcs],
            'text_cnt': priors['text_cnt'].get(text, 0),
            'mc_lem_mean': np.mean(lemma_p, axis=0) if lemma_p else 0.0,
            'mc_lem_max': np.max(lemma_p, axis=0) if lemma_p else 0.0,
            'mc_vid_p': [priors['vid'].get((filt.get('Вид услуги'), m), np.nan) for m in mcs],
            'vid_match': [float(f'Вид услуги {filt["Вид услуги"]}' in it_params[r]) if 'Вид услуги' in filt else np.nan for r in rows],
            'type_match': [float(f'Тип услуги {filt["Тип услуги"]}' in it_params[r]) if 'Тип услуги' in filt else np.nan for r in rows],
            'title_overlap': [len(q_lem & title_lemmas[r]) / max(len(q_lem), 1) for r in rows],
            'q_len': len(q_lem), 'has_filters': int(bool(filt)), 'pool_size': len(c['rows']),
        })
        f[ITEM_NUM_COLS] = item_num[rows]
        if relevant is not None:
            f['target'] = np.isin(item_id_arr[rows], list(relevant[key])).astype(np.int8)
        frames.append(f)
    return pd.concat(frames, ignore_index=True)

t0 = time.time()
val_feat = ranker_features(val_cands, val_queries, val_queries.query_key, fit_priors, cooc_share.to_dict(), val_relevant)
RANKER_FEATURES = [c for c in val_feat.columns if c not in ('key', 'row', 'target')]
in_pool = val_feat.groupby('key').target.sum() / val_relevant.str.len()
print(f'Признаки: {val_feat.shape}, {time.time() - t0:.0f} с, доля релевантных в пуле: {in_pool.mean():.4f}')

Признаки: (954966, 30), 34 с, доля релевантных в пуле: 0.9288


**Качество на двух фолдах**

Запросы валидации делим на две половины: обучаем на одной, считаем Recall50 на другой и наоборот.

In [73]:
RANKER_PARAMS = dict(objective='lambdarank', learning_rate=0.05, num_leaves=31, min_child_samples=50,
                     subsample=0.8, subsample_freq=1, colsample_bytree=0.8, lambdarank_truncation_level=60,
                     n_estimators=300, random_state=SEED, deterministic=True, force_row_wise=True,
                     n_jobs=8, verbose=-1)

def fit_ranker(feat):
    model = lgb.LGBMRanker(**RANKER_PARAMS)
    model.fit(feat[RANKER_FEATURES], feat.target, group=feat.groupby('key', sort=False).size().to_numpy())
    return model

def ranker_top(model, feat, k=50):
    scored = feat[['key', 'row']].assign(score=model.predict(feat[RANKER_FEATURES]))
    return {key: item_id_arr[g.row.to_numpy()[np.argsort(-g.score.to_numpy(), kind='stable')[:k]]].tolist()
            for key, g in scored.groupby('key', sort=False)}

fold_of = dict(zip(val_queries.query_key, np.random.default_rng(SEED).permutation(len(val_queries)) % 2))
val_feat['fold'] = val_feat.key.map(fold_of)
oof_preds = {}
for fold in (0, 1):
    model = fit_ranker(val_feat[val_feat.fold != fold])
    oof_preds.update(ranker_top(model, val_feat[val_feat.fold == fold]))

pd.Series({
    'BM25': recall_at_k(bm25_top(val_cands, **BM25_PARAMS), val_relevant),
    'BM25 30 + векторный 20': recall_at_k(union_top(val_cands, 20), val_relevant),
    'ранкер': recall_at_k(oof_preds, val_relevant),
}, name='Recall50').round(4)

BM25                      0.8516
BM25 30 + векторный 20    0.8833
ранкер                    0.8891
Name: Recall50, dtype: float64

In [74]:
importance = pd.Series(model.booster_.feature_importance('gain'), RANKER_FEATURES)
(importance / importance.sum()).sort_values(ascending=False).head(10).round(3)

bm25_rank        0.420
bm25_text_rel    0.192
dense_rank       0.085
dense_rel        0.071
desc_len         0.024
mc_lem_mean      0.022
dense            0.021
reviews          0.019
bm25_text        0.018
rating           0.016
dtype: float64

In [78]:
# CatBoost только для сравнения, итоговый ответ строит LightGBM
from catboost import CatBoostRanker, Pool

cb_preds = {}
for fold in (0, 1):
    tr, te = val_feat[val_feat.fold != fold], val_feat[val_feat.fold == fold]
    cb_model = CatBoostRanker(loss_function='YetiRank', iterations=500, learning_rate=0.1, depth=6,
                              random_seed=SEED, thread_count=8, verbose=0,
                              allow_writing_files=False)
    cb_model.fit(Pool(tr[RANKER_FEATURES], tr.target, group_id=tr.key.to_numpy()))
    cb_preds.update(ranker_top(cb_model, te))

pd.Series({
    'LightGBM lambdarank': recall_at_k(oof_preds, val_relevant),
    'CatBoost YetiRank': recall_at_k(cb_preds, val_relevant),
}, name='Recall50').round(4)

LightGBM lambdarank    0.8891
CatBoost YetiRank      0.8842
Name: Recall50, dtype: float64

## Итоговый ответ на бенчмарк

Итоговый ранкер обучаем на всех запросах валидации. Для бенчмарка пул строим по корпусу бенчмарка, соседство и вероятности подкатегорий - по всему train. Файл `answer.csv` перезаписывается ответом ранкера.

In [76]:
final_ranker = fit_ranker(val_feat)
add_dense_scores(bench_cands, queries, queries.query_id)
bench_feat = ranker_features(bench_cands, queries, queries.query_id, full_priors, full_share.to_dict())
bench_preds = ranker_top(final_ranker, bench_feat)
save_answer(bench_preds, 'answer.csv')
check_answer('answer.csv')

answer.csv: формат верный, строк 2452, объявлений в строке в среднем 50.0


**Выводы по ранкеру**

- Ранкер на объединённом пуле даёт `Recall50=0.8891` против `0.8833` у простого объединения и `0.8516` у BM25.
- Главные признаки - места кандидата в BM25 и векторном поиске и их относительные скоры, дальше идут вероятность подкатегории по леммам запроса, отзывы и рейтинг.
- В пуле из 400 кандидатов лежит `92.9%` релевантных объявлений, так что запас для улучшения ранкера ещё есть.